In [1]:
!pip install pympler cloudpickle ucimlrepo

In [2]:
import io,  time, warnings
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

from RuleTree.tree.TrepanClassifier import TrepanClassifier
from RuleTree.tree.TrepanClassifierOptimized import TrepanClassifierOptimized
from RuleTree.tree.RuleTreeClassifier import RuleTreeClassifier
from RuleTree.stumps.classification import DecisionTreeStumpClassifier
from RuleTree.utils.memory_utils import deep_sklearn_sizeof, count_tree_structure

from pympler import asizeof
import cloudpickle

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
# 1. Carica Heart Disease (UCI id=45) 
heart_disease = fetch_ucirepo(id=45)
X = heart_disease.data.features
y = heart_disease.data.targets

#  2. Mode imputation per i NaN (ca, thal) 
X = X.copy()
X['ca']   = X['ca'].fillna(X['ca'].mode()[0])
X['thal'] = X['thal'].fillna(X['thal'].mode()[0])

#  3. Target binario: 0 = no disease, 1 = disease 
y_binary = (y > 0).astype(int)
y = y_binary.values.ravel()

# 4. Categorical features
categorical_features_names = ['sex','cp','fbs','restecg','exang','slope','thal']
categorical_features = [X.columns.get_loc(col) for col in categorical_features_names]
attributes = list(X.columns)

# 5. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. StandardScaler 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)



print(f"X_train {X_train.shape} | X_test {X_test.shape}")
print(f"y_train {np.bincount(y_train)} | y_test {np.bincount(y_test)}")
print(f"categorical_features (indici): {categorical_features}")

X_train (242, 13) | X_test (61, 13)
y_train [131 111] | y_test [33 28]
categorical_features (indici): [1, 2, 5, 6, 8, 10, 12]


In [4]:
CURRENT_MAX_INTERNAL = 5    # <-- parametro principale
S_MIN = 300

# RuleTree non ha max_internal_nodes: approssimiamo con max_leaf_nodes
MAX_LEAF = CURRENT_MAX_INTERNAL + 1 #CURRENT_MAX_INTERNAL + 1  

RESULTS_FILE = "risultati_accumulati_heart.json"

import os, json
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        results = json.load(f)
    print(f"Caricati {len(results)} risultati da {RESULTS_FILE}")
else:
    results = []
    print("Nessun file precedente, parto da zero")

Caricati 24 risultati da risultati_accumulati_heart.json


In [5]:
def get_params_for(cls, oracle):
    if cls is RuleTreeClassifier:
        return dict(
            max_leaf_nodes=MAX_LEAF,      # <-- approssima max_internal_nodes
            random_state=42,
            base_stumps=DecisionTreeStumpClassifier(max_depth=1),
        )
    else:  # TrepanClassifier / TrepanClassifierOptimized
        return dict(
            estimator=oracle,
            max_internal_nodes=CURRENT_MAX_INTERNAL,   # <-- parametro principale
            s_min=S_MIN,
            epsilon=0.01,
            delta=0.01,
            random_state=42,
            categorical_features=categorical_features,
        )

In [6]:
# --- nbytes (dati puri) ---
dataset_nbytes = X_train.nbytes + y_train.nbytes
print(f"Dataset (nbytes):      {dataset_nbytes/1024/1024:.2f} MB")

Dataset (nbytes):      0.02 MB


In [7]:
# --- Oracolo: MLP come in heart_disease.ipynb ---
oracle = MLPClassifier(hidden_layer_sizes=(), max_iter=500, random_state=42)
oracle.fit(X_train, y_train)
print(f"Oracolo MLP accuracy test: {oracle.score(X_test, y_test):.4f}")

for name, cls in [("base", RuleTreeClassifier),
                  ("Ottimizzato", TrepanClassifierOptimized),
                  ("Originale", TrepanClassifier)]:
    print(f"\n{'='*60}\n{name}\n{'='*60}")

    params = get_params_for(cls, oracle)
    model = cls(**params)

    t0 = time.time()
    model.fit(X_train, y_train)
    t = time.time() - t0

    asizeof_kb = asizeof.asizeof(model.root) / 1024
    deep_kb    = deep_sklearn_sizeof(model.root) / 1024

    buffer = io.BytesIO()
    cloudpickle.dump(model.root, buffer)
    cloudpickle_kb = buffer.tell() / 1024
    buffer.close()

    struct = count_tree_structure(model.root)
    acc = model.score(X_test, y_test)
    fidelity = (model.predict(X_test) == oracle.predict(X_test)).mean()

    print(f"cloudpickle:     {cloudpickle_kb:.2f} KB")
    print(f"Accuracy:        {acc:.4f}")
    print(f"Fidelity:        {fidelity:.4f}")
    print(f"asizeof:         {asizeof_kb:.1f} KB")
    print(f"deep_sizeof:     {deep_kb:.1f} KB")
    print(f"Tempo:           {t:.2f} s")

    results.append({
        "max_internal_nodes": CURRENT_MAX_INTERNAL,
        "max_leaf_nodes":     MAX_LEAF if cls is RuleTreeClassifier else None,
        "s_min":              S_MIN,
        "name":               name,
        "accuracy":           acc,
        "fidelity":           fidelity,
        "asizeof_kb":         asizeof_kb,
        "deep_kb":            deep_kb,
        "cloudpickle_kb":     cloudpickle_kb,
        "n_nodes":            struct['n_nodes'],
        "n_stumps":           struct['n_stumps'],
        "n_leaves":           struct['n_leaves'],
        "n_conditions":       struct['n_conditions'],
        "n_rules_copies":     struct['n_rules_copies'],
        "time_s":             t,
    })

    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2)

Oracolo MLP accuracy test: 0.8689

base
cloudpickle:     6.42 KB
Accuracy:        0.8361
Fidelity:        0.8361
asizeof:         22.7 KB
deep_sizeof:     20.4 KB
Tempo:           0.01 s

Ottimizzato
cloudpickle:     4.36 KB
Accuracy:        0.7541
Fidelity:        0.8852
asizeof:         18.5 KB
deep_sizeof:     17.0 KB
Tempo:           0.49 s

Originale
cloudpickle:     6.03 KB
Accuracy:        0.7541
Fidelity:        0.8852
asizeof:         22.3 KB
deep_sizeof:     20.4 KB
Tempo:           0.51 s


In [8]:
import pandas as pd, json

with open(RESULTS_FILE) as f:
    results_all = json.load(f)

df = pd.DataFrame(results_all)
df = df.drop_duplicates(subset=["max_internal_nodes", "name"], keep="last").reset_index(drop=True)

df = df[[
    "max_internal_nodes", "name",
    "accuracy", "fidelity",
    "asizeof_kb", "deep_kb", "cloudpickle_kb",
    "n_stumps", "n_leaves", "n_rules_copies", "time_s",
]]

# Ordina mettendo i None/NaN in fondo
df = df.sort_values(
    ["max_internal_nodes", "name"],
    kind="stable",
    na_position="last",
).reset_index(drop=True)

def fmt(x, d=3):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "--"
    return f"{x:.{d}f}"

rows = []
prev = None
for _, r in df.iterrows():
    key = r["max_internal_nodes"]

    # \midrule quando cambia il gruppo (gestisce anche NaN)
    if prev is not None and not (
        (pd.isna(prev) and pd.isna(key)) or (prev == key)
    ):
        rows.append("\\midrule")

    # Colonna Int. nodes: "--" se NaN/None, altrimenti intero
    if pd.isna(key):
        int_str = "--"
    else:
        int_str = str(int(key))

    rows.append(
        f"{int_str} & {r['name']:<11} & "
        f"{fmt(r['accuracy'])} & {fmt(r['fidelity'])} & "
        f"{fmt(r['asizeof_kb'])} & {fmt(r['deep_kb'])} & "
        f"{fmt(r['cloudpickle_kb'])} & "
        f"{int(r['n_stumps'])} & {int(r['n_leaves'])} & {int(r['n_rules_copies'])} & "
        f"{fmt(r['time_s'])} \\\\"
    )
    prev = key

body = "\n".join(rows)

latex = r"""\documentclass{article}
\usepackage[a4paper, left=1cm, right=2.5cm, top=2.5cm, bottom=2.5cm]{geometry}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{makecell}

\title{Risultati memoria}
\author{Davide Catena}
\date{September 2026}

\begin{document}
\maketitle

\section{Heart-disease Dataset}

\begin{table}[htbp]
\raggedright
\caption{Confronto TREPAN (originale e ottimizzato) e RuleTree base su Heart Disease al variare di \texttt{max\_internal\_nodes}. RuleTree usa \texttt{max\_leaf\_nodes = max\_internal\_nodes + 1}.}
\label{tab:risultati}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{c l r r r r r r r r r}
\toprule
\makecell{Int.\\nodes} & Model & Acc. & Fid. & asizeof & deep & cpickle & Stumps & Leaves & Rules & Time \\
 & & & & (KB) & (KB) & (KB) & & & & (s)  \\
\midrule
""" + body + r"""
\bottomrule
\end{tabular}
\end{table}

\end{document}
"""

with open("risultati_heart.tex", "w") as f:
    f.write(latex)

print(latex)

\documentclass{article}
\usepackage[a4paper, left=1cm, right=2.5cm, top=2.5cm, bottom=2.5cm]{geometry}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{makecell}

\title{Risultati memoria}
\author{Davide Catena}
\date{September 2026}

\begin{document}
\maketitle

\section{Heart-disease Dataset}

\begin{table}[htbp]
\raggedright
\caption{Confronto TREPAN (originale e ottimizzato) e RuleTree base su Heart Disease al variare di \texttt{max\_internal\_nodes}. RuleTree usa \texttt{max\_leaf\_nodes = max\_internal\_nodes + 1}.}
\label{tab:risultati}
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{c l r r r r r r r r r}
\toprule
\makecell{Int.\\nodes} & Model & Acc. & Fid. & asizeof & deep & cpickle & Stumps & Leaves & Rules & Time \\
 & & & & (KB) & (KB) & (KB) & & & & (s)  \\
\midrule
3 & Originale   & 0.754 & 0.885 & 15.180 & 13.984 & 4.067 & 3 & 4 & 12 & 0.215 \\
3 & Ottimizzato & 0.754 & 0.885 & 12.789 & 11.844 & 3.000 & 3 & 4 & 6 & 0.774 \\
3 & base        & 0.770 & 0.7